[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/JohnSnowLabs/spark-nlp/blob/master/examples/python/benchmarks/Benchmark_WordSegmentation.ipynb)

# Benchmark: Word Segmentation

Scores a pretrained Chinese word segmenter's boundary-level precision/recall/F1 against
gold-labeled data, using `sparknlp.benchmark.Benchmark.evaluate(..., task="wordsegmentation")`.
Word segmentation only makes sense as a benchmark for languages that aren't already
whitespace-delimited (Chinese, Japanese, Thai, ...) -- for English, a `Tokenizer` is trivially
~100% correct, so it isn't a meaningful benchmark target the way it is here.

**Dataset**: [Universal Dependencies Chinese GSD](https://github.com/UniversalDependencies/UD_Chinese-GSD),
test split (CC BY-SA 4.0) -- every token is marked `SpaceAfter=No`, i.e. Chinese text here is
written with no spaces between words, and the gold segmentation is exactly the token boundaries
UD annotated.

**Model**: `WordSegmenterModel.pretrained()` (default: `wordseg_pku`), trained on the PKU
(Peking University) segmentation standard -- a different, though related, segmentation
convention than UD's own.

## Setup

Run these cells first on a fresh Colab runtime.

In [3]:
!wget https://setup.johnsnowlabs.com/colab.sh -O - | bash

--2026-08-29 10:57:08--  https://setup.johnsnowlabs.com/colab.sh
Resolving setup.johnsnowlabs.com (setup.johnsnowlabs.com)... 3.86.22.73
Connecting to setup.johnsnowlabs.com (setup.johnsnowlabs.com)|3.86.22.73|:443... connected.
HTTP request sent, awaiting response... 302 Moved Temporarily
Location: https://raw.githubusercontent.com/JohnSnowLabs/spark-nlp/master/scripts/colab_setup.sh [following]
--2026-08-29 10:57:08--  https://raw.githubusercontent.com/JohnSnowLabs/spark-nlp/master/scripts/colab_setup.sh
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.109.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 1483 (1.4K) [text/plain]
Saving to: ‘STDOUT’


-                     0%[                    ]       0  --.-KB/s               
-                   100%[===================>]   1.45K  --.-KB/s    in 0s      



In [4]:
# Current Colab runtimes default to Java 21, which Spark 3.4.x (what the bootstrap above
# installs) isn't compatible with -- Spark's low-level Platform.java reflection breaks on it.
# Switch to Java 17, which Spark 3.4.x does support.
!apt-get update -qq && apt-get install -y -qq openjdk-17-jdk-headless
import os
os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-17-openjdk-amd64"
os.environ["PATH"] = os.environ["JAVA_HOME"] + "/bin:" + os.environ["PATH"]

In [5]:
# Current Colab runtimes also default to Python 3.13, which removed the deprecated
# `typing.io` submodule -- but Spark 3.4.x's own source still does `from typing.io import
# BinaryIO`. This patches that one import, both for this notebook process and for the
# separate Python worker subprocesses Spark launches to actually run distributed tasks
# (those load pyspark from its own bundled zip, so both copies need patching).
import zipfile, shutil

site_pkgs = "/usr/local/lib/python3.13/dist-packages"
loose_path = f"{site_pkgs}/pyspark/broadcast.py"
zip_path = f"{site_pkgs}/pyspark/python/lib/pyspark.zip"
OLD = "from typing.io import BinaryIO  # type: ignore[import]"
NEW = "from typing import BinaryIO  # patched for Python 3.13 (typing.io removed)"

with open(loose_path) as f:
    text = f.read()
with open(loose_path, "w") as f:
    f.write(text.replace(OLD, NEW))

tmp_path = zip_path + ".tmp"
with zipfile.ZipFile(zip_path, "r") as zin, zipfile.ZipFile(tmp_path, "w", zipfile.ZIP_DEFLATED) as zout:
    for item in zin.infolist():
        data = zin.read(item.filename)
        if item.filename == "pyspark/broadcast.py":
            data = data.decode("utf-8").replace(OLD, NEW).encode("utf-8")
        zout.writestr(item, data)
shutil.move(tmp_path, zip_path)
print("Environment patched for this Colab runtime (Java 17, typing.io).")

Environment patched for this Colab runtime (Java 17, typing.io).

In [6]:
import sparknlp
spark = sparknlp.start()
print("Spark NLP version:", sparknlp.version())
print("Apache Spark version:", spark.version)
from sparknlp.base import DocumentAssembler
from sparknlp.annotator import WordSegmenterModel
from pyspark.ml import Pipeline
from sparknlp.benchmark import Benchmark

Spark NLP version: 6.4.2
Apache Spark version: 3.4.4

## 1. Get some data

In [8]:
import urllib.request

url = "https://raw.githubusercontent.com/UniversalDependencies/UD_Chinese-GSD/master/zh_gsd-ud-test.conllu"
conllu_text = urllib.request.urlopen(url, timeout=30).read().decode("utf-8")

sentences = []
current = []
for line in conllu_text.split("\n"):
    if line.startswith("#"):
        continue
    if line.strip() == "":
        if current:
            sentences.append(current)
            current = []
        continue
    cols = line.split("\t")
    if "-" in cols[0] or "." in cols[0]:
        continue  # multiword-token / empty-node lines
    current.append(cols[1])  # FORM
if current:
    sentences.append(current)

print(len(sentences), "sentences")
print(sentences[0])

500 sentences
['然而', '，', '這樣', '的', '處理', '也', '衍生', '了', '一些', '問題', '。']

> **Note: Spark NLP's begin/end offsets are inclusive on both ends** -- `end` is the index
> of the *last* character, not one past it (e.g. a 2-character token starting at 0 has
> `begin=0, end=1`). This is different from Python's usual half-open convention, and
> `Benchmark.evaluate`'s `wordsegmentation` gold `"begin:end"` strings need to use the same
> inclusive convention to match what it reads off the pipeline's own predictions -- getting this
> backwards silently produces near-zero scores even when the model's segmentation is correct.

In [10]:
rows = []
for forms in sentences:
    text = "".join(forms)  # every token here is SpaceAfter=No -- no separator between words
    boundaries = []
    pos = 0
    for form in forms:
        boundaries.append(f"{pos}:{pos + len(form) - 1}")  # inclusive end, matches Spark NLP
        pos += len(form)
    rows.append((text, boundaries))

gold_data = spark.createDataFrame(rows, ["text", "label"])
gold_data.show(3, truncate=60)

+--------------------------------------------------------------------------------+------------------------------------------------------------+
|                                                                            text|                                                       label|
+--------------------------------------------------------------------------------+------------------------------------------------------------+
|                                              然而，這樣的處理也衍生了一些問題。|[0:1, 2:2, 3:4, 5:5, 6:7, 8:8, 9:10, 11:11, 12:13, 14:15,...|
|                    自從2004年提出了興建人文大樓的構想，企業界陸續有人提供捐款。|[0:1, 2:5, 6:6, 7:8, 9:9, 10:11, 12:13, 14:15, 16:16, 17:...|
|杜鵑花為溫帶植物，台北雖然在亞熱帶，但冬季的東北季風卻使得杜鵑花在臺大宜然自得。|[0:2, 3:3, 4:5, 6:7, 8:8, 9:10, 11:12, 13:13, 14:14, 15:1...|
+--------------------------------------------------------------------------------+------------------------------------------------------------+
only showing top 3 rows

## 2. Build the pipeline

In [12]:
document_assembler = DocumentAssembler().setInputCol("text").setOutputCol("document")
word_segmenter = WordSegmenterModel.pretrained() \
    .setInputCols(["document"]).setOutputCol("token")

pipeline = Pipeline(stages=[document_assembler, word_segmenter])
pipeline_model = pipeline.fit(gold_data)

pipeline_model.transform(gold_data.limit(1)).selectExpr("text", "token.result").show(truncate=False)

wordseg_pku download started this may take some time.
Approximate size to download 2 MB

[ | ]
[ / ]
[OK!]
+----------------------------------+--------------------------------------------------------+
|text                              |result                                                  |
+----------------------------------+--------------------------------------------------------+
|然而，這樣的處理也衍生了一些問題。|[然而, ，, 這樣, 的, 處理, 也, 衍生, 了, 一些, 問題, 。]|
+----------------------------------+--------------------------------------------------------+

## 3. Run the benchmark

In [14]:
report = Benchmark.evaluate(pipeline_model, gold_data, task="wordsegmentation", label_col="label")
print(report)

wordsegmentation accuracy (n=12010): f1=0.6563, precision=0.6376, recall=0.6762
  segment: f1=0.6563, precision=0.6376, recall=0.6762

## Reading the result

PKU's segmentation convention isn't identical to UD's (e.g. treatment of names, compounds), so
some disagreement here is expected and meaningful, not a bug.